# Summary Statistics from RF Predictions

This notebook reads per-country RF prediction GPKGs, applies quality filtering, and
aggregates deprivation statistics at city, country, and regional level.

**Inputs:** Per-country RF prediction GeoPackages in `data_external/zenodo/predictions/`  
— produced by `01_apply_rf_predictions.ipynb`, also available from the Zenodo deposit  
([DOI: 10.5281/zenodo.18788260](https://doi.org/10.5281/zenodo.18788260)).  
These files are large and intentionally excluded from GitHub.

Required columns per GPKG:
- `REG1_GHSL` — region label (e.g. *Africa*, *Asia*, *Latin America and the Caribbean*)
- `UC_NM_MN` — city name
- `rf_label` — 0/1 (non-deprived / deprived segment)
- `POP_SEG` — segment population

**Quality filter:** cities where fewer than 80% of segments have valid `rf_label`
and `POP_SEG` are excluded from all downstream aggregations.

**City size classification** follows the UN *World Urbanization Prospects 2018* scheme:

| Class | Population |
|---|---|
| Small | < 500,000 |
| Medium | 500,000–999,999 |
| Large | 1,000,000–4,999,999 |
| Very large | 5,000,000–9,999,999 |
| Megacity | ≥ 10,000,000 |

**Outputs:** Summary CSVs are small and committed to the repository under
`2_modelling/02_application/summary_statistics/`.

> **Note on populations:** City-level population figures here are derived from
> segment-aggregated `POP_SEG` values in the RF prediction GPKGs. The separate
> UCDB/GHS-POP coverage and omission analysis (Revision 2) uses independently
> computed UCDB polygon populations and is handled in `notebooks/revision2_coverage/`.

# Imports and paths

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATH CONFIGURATION  (portable — no hard-coded local paths)
# ============================================================
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "citysegmentdeprivation" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_EXTERNAL = REPO_ROOT / "data_external"
ZENODO_DATA   = DATA_EXTERNAL / "zenodo"

# Per-country RF prediction GPKGs (large — not committed to GitHub)
PREDICTIONS_FOLDER = ZENODO_DATA / "predictions"

# Summary statistics output folder (small CSVs — committed to GitHub)
SUMMARY_DIR = REPO_ROOT / "2_modelling" / "02_application" / "summary_statistics"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

print("PREDICTIONS_FOLDER:", PREDICTIONS_FOLDER.resolve())
print("SUMMARY_DIR:", SUMMARY_DIR.resolve())

# Parameters and helpers

In [2]:
# Quality requirement: at least this % of blocks in a city must have valid rf_label and POP_SEG
VALID_BLOCKS_THRESHOLD = 80.0  # %

def classify_city_size(pop):
    """UN WUP 2018 city size classes based on city population."""
    if pop < 500_000:
        return "Small"
    elif pop < 1_000_000:
        return "Medium"
    elif pop < 5_000_000:
        return "Large"
    elif pop < 10_000_000:
        return "Very large"
    else:
        return "Megacity"


# City level deprivation stats (with 80% QC)

In [3]:
records = []

for file in PREDICTIONS_FOLDER.glob("*_rf_preds.gpkg"):
    country = file.stem.replace("_rf_preds", "")
    
    try:
        gdf = gpd.read_file(
            file,
            columns=["REG1_GHSL", "UC_NM_MN", "rf_label", "POP_SEG"]
        )
    except Exception as e:
        print(f"⚠️ Skipping {file.name} due to error: {e}")
        continue

    # valid block = both rf_label and POP_SEG present
    gdf["valid"] = ~(gdf["rf_label"].isna() | gdf["POP_SEG"].isna())

    for city, sub in gdf.groupby("UC_NM_MN"):
        total_blocks = len(sub)
        valid_blocks = int(sub["valid"].sum())
        pct_valid = (valid_blocks / total_blocks * 100) if total_blocks > 0 else 0.0

        if pct_valid < VALID_BLOCKS_THRESHOLD:
            # city fails quality threshold – excluded from downstream stats
            continue

        sub_valid = sub[sub["valid"]]

        total_pop = float(sub_valid["POP_SEG"].sum())
        deprived_pop = float(
            sub_valid.loc[sub_valid["rf_label"] == 1, "POP_SEG"].sum()
        )
        pct_deprived = (deprived_pop / total_pop * 100) if total_pop > 0 else 0.0

        records.append({
            "Region": sub["REG1_GHSL"].iloc[0],
            "Country": country,
            "City": city,
            "TotalBlocks": total_blocks,
            "ValidBlocks": valid_blocks,
            "PctValid": pct_valid,
            "TotalPop": total_pop,
            "DeprivedPop": deprived_pop,
            "PctDeprived": pct_deprived,
        })

city_df = pd.DataFrame(records)
print("Number of cities passing QC (>= 80% valid blocks):", len(city_df))

# Save core city-level deprivation table
city_deprivation_csv = SUMMARY_DIR / "city_deprivation_80pct.csv"
city_df.to_csv(city_deprivation_csv, index=False, encoding="utf-8")
city_df.head()


Number of cities passing QC (>= 80% valid blocks): 5142


,Region,Country,City,TotalBlocks,ValidBlocks,PctValid,TotalPop,DeprivedPop,PctDeprived
0,Asia,afghanistan,Bagram,19,19,100.0,98120.043719,0.000000,0.000000
1,Asia,afghanistan,Chaman,63,63,100.0,175884.146438,0.000000,0.000000
2,Asia,afghanistan,Charikar,203,203,100.0,182398.355785,17887.717447,9.806951
3,Asia,afghanistan,Farah,124,124,100.0,135327.620853,0.000000,0.000000
4,Asia,afghanistan,Guzarah,71,71,100.0,164797.329928,0.000000,0.000000


# All cities pass the qualicy check.

In [4]:
# City size classification based on TotalPop (UN WUP 2018 scheme)
city_df["CitySizeClass"] = city_df["TotalPop"].apply(classify_city_size)

# Reordered view for manuscript tables / figures
city_table = city_df[[
    "Region", "Country", "City", "CitySizeClass",
    "TotalPop", "DeprivedPop", "PctDeprived"
]].copy()

city_with_size_csv = SUMMARY_DIR / "city_deprivation_with_sizeclass_80pct.csv"
city_table.to_csv(city_with_size_csv, index=False, encoding="utf-8")

city_table.head()


,Region,Country,City,CitySizeClass,TotalPop,DeprivedPop,PctDeprived
0,Asia,afghanistan,Bagram,Small,98120.043719,0.000000,0.000000
1,Asia,afghanistan,Chaman,Small,175884.146438,0.000000,0.000000
2,Asia,afghanistan,Charikar,Small,182398.355785,17887.717447,9.806951
3,Asia,afghanistan,Farah,Small,135327.620853,0.000000,0.000000
4,Asia,afghanistan,Guzarah,Small,164797.329928,0.000000,0.000000


## Country level aggregation

In [5]:
country_df = (
    city_df.groupby(["Region", "Country"], as_index=False)
    .agg(
        TotalPop=("TotalPop", "sum"),
        DeprivedPop=("DeprivedPop", "sum")
    )
)
country_df["PctDeprived"] = (
    country_df["DeprivedPop"] / country_df["TotalPop"] * 100
)

country_csv = SUMMARY_DIR / "country_deprivation_80pct.csv"
country_df.to_csv(country_csv, index=False, encoding="utf-8")
country_df.head()


,Region,Country,TotalPop,DeprivedPop,PctDeprived
0,Africa,algeria,1.932588e+07,2.148402e+06,11.116709
1,Africa,angola,2.065083e+07,1.261948e+07,61.108829
2,Africa,benin,4.673286e+06,1.395721e+06,29.865954
3,Africa,botswana,5.494540e+05,5.871468e+04,10.686005
4,Africa,burkina_faso,5.103501e+06,3.050541e+06,59.773491


## Region-level aggregation

In [6]:
region_df = (
    city_df.groupby("Region", as_index=False)
    .agg(
        TotalPop=("TotalPop", "sum"),
        DeprivedPop=("DeprivedPop", "sum"),
        Countries=("Country", "nunique"),
        Cities=("City", "nunique")
    )
)
region_df["PctDeprived"] = (
    region_df["DeprivedPop"] / region_df["TotalPop"] * 100
)

regional_csv = SUMMARY_DIR / "regional_deprivation_80pct.csv"
region_df.to_csv(regional_csv, index=False, encoding="utf-8")
region_df


,Region,TotalPop,DeprivedPop,Countries,Cities,PctDeprived
0,Africa,5.119094e+08,2.230723e+08,52,1484,43.576512
1,Asia,1.108310e+09,1.253191e+08,30,2675,11.307226
2,Europe,7.759970e+05,0.000000e+00,1,5,0.000000
3,Latin America and the Caribbean,3.383702e+08,4.655389e+07,21,936,13.758269
4,Oceania,8.940537e+05,0.000000e+00,3,5,0.000000


## Region x size-class deprivation

In [7]:
# Region × SizeClass breakdown
size_summary = (
    city_df.groupby(["Region", "CitySizeClass"], as_index=False)
    .agg(
        Cities=("City", "nunique"),
        TotalPop=("TotalPop", "sum"),
        DeprivedPop=("DeprivedPop", "sum")
    )
)
size_summary["PctDeprived"] = (
    size_summary["DeprivedPop"] / size_summary["TotalPop"] * 100
)

region_sizeclass_csv = SUMMARY_DIR / "region_sizeclass_deprivation_80pct.csv"
size_summary.to_csv(region_sizeclass_csv, index=False, encoding="utf-8")
size_summary


,Region,CitySizeClass,Cities,TotalPop,DeprivedPop,PctDeprived
0,Africa,Large,69,1.486244e+08,6.656293e+07,44.785994
1,Africa,Medium,77,5.249917e+07,2.103587e+07,40.068953
2,Africa,Megacity,4,6.113634e+07,3.520470e+07,57.583926
3,Africa,Small,1325,1.842288e+08,6.303350e+07,34.214793
4,Africa,Very large,10,6.542066e+07,3.723526e+07,56.916665
5,Asia,Large,132,2.675109e+08,2.642951e+07,9.879787
6,Asia,Medium,147,1.033018e+08,9.123031e+06,8.831432
7,Asia,Megacity,13,2.850077e+08,5.232733e+07,18.359971
8,Asia,Small,2372,3.470752e+08,2.517054e+07,7.252185
9,Asia,Very large,15,1.054147e+08,1.226875e+07,11.638558


## Region-country-city list (no QC, all cities)

In [8]:
# Simple region–country–city inventory based on geometries
records_all = []

for file in PREDICTIONS_FOLDER.glob("*_rf_preds.gpkg"):
    country = file.stem.replace("_rf_preds", "")
    
    try:
        gdf = gpd.read_file(file, columns=["REG1_GHSL", "UC_NM_MN"])
    except Exception as e:
        print(f"⚠️ Skipping {file.name} due to error: {e}")
        continue

    gdf = gdf.drop_duplicates(subset=["UC_NM_MN"])

    for _, row in gdf.iterrows():
        records_all.append({
            "Region": row["REG1_GHSL"],
            "Country": country,
            "City": row["UC_NM_MN"]
        })

detailed_df = pd.DataFrame(records_all)

region_country_city_csv = SUMMARY_DIR / "region_country_city.csv"
detailed_df.to_csv(region_country_city_csv, index=False, encoding="utf-8")
detailed_df.head()


,Region,Country,City
0,Asia,afghanistan,Bagram
1,Asia,afghanistan,Chaman
2,Asia,afghanistan,Charikar
3,Asia,afghanistan,Farah
4,Asia,afghanistan,Guzarah


## Detailed city classification (80% QC) with size class

In [9]:
# Detailed classification table for cities passing the 80% QC
filtered_df = city_df[[
    "Region", "Country", "City", "TotalPop"
]].copy()
filtered_df["SizeClass"] = filtered_df["TotalPop"].apply(classify_city_size)
filtered_df.rename(columns={"TotalPop": "Population"}, inplace=True)

region_country_city_sizeclass_csv = (
    SUMMARY_DIR / "region_country_city_sizeclass_80pct.csv"
)
filtered_df.to_csv(region_country_city_sizeclass_csv, index=False, encoding="utf-8")
filtered_df.head()


,Region,Country,City,Population,SizeClass
0,Asia,afghanistan,Bagram,98120.043719,Small
1,Asia,afghanistan,Chaman,175884.146438,Small
2,Asia,afghanistan,Charikar,182398.355785,Small
3,Asia,afghanistan,Farah,135327.620853,Small
4,Asia,afghanistan,Guzarah,164797.329928,Small


## Region summary with city counts & population by size class

In [10]:
def count_size(x, label):
    return (x == label).sum()

def pop_size(df, label):
    return df.loc[df["SizeClass"] == label, "Population"].sum()

# Base region summary
region_summary = (
    filtered_df.groupby("Region", as_index=False)
    .agg(
        Countries=("Country", "nunique"),
        Cities=("City", "nunique"),
        TotalPopulation=("Population", "sum"),
    )
)

# Compute size-class specific counts and populations
labels = ["Small", "Medium", "Large", "Very large", "Megacity"]

grouped = filtered_df.groupby("Region")
for label in labels:
    region_summary[f"{label}_Cities"] = grouped["SizeClass"].apply(
        lambda x, lbl=label: count_size(x, lbl)
    ).values
    region_summary[f"{label}_Pop"] = grouped.apply(
        lambda df, lbl=label: pop_size(df, lbl)
    ).values

# Optional: population in millions for readability
region_summary["TotalPop_Millions"] = (region_summary["TotalPopulation"] / 1_000_000).round(2)
for label in labels:
    region_summary[f"{label}_Pop_Millions"] = (
        region_summary[f"{label}_Pop"] / 1_000_000
    ).round(2)

region_summary_csv = SUMMARY_DIR / "region_summary_sizeclass_population_80pct.csv"
region_summary.to_csv(region_summary_csv, index=False, encoding="utf-8")
region_summary


C:\Users\saigveer\AppData\Local\Temp\ipykernel_24348\2461208178.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  region_summary[f"{label}_Pop"] = grouped.apply(
C:\Users\saigveer\AppData\Local\Temp\ipykernel_24348\2461208178.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  region_summary[f"{label}_Pop"] = grouped.apply(
C:\Users\saigveer\AppData\Local\Temp\ipykernel_24348\2461208178.py:25: FutureWarning: Data

,Region,Countries,Cities,TotalPopulation,Small_Cities,Small_Pop,Medium_Cities,Medium_Pop,Large_Cities,Large_Pop,Very large_Cities,Very large_Pop,Megacity_Cities,Megacity_Pop,TotalPop_Millions,Small_Pop_Millions,Medium_Pop_Millions,Large_Pop_Millions,Very large_Pop_Millions,Megacity_Pop_Millions
0,Africa,52,1484,5.119094e+08,1332,1.842288e+08,77,5.249917e+07,69,1.486244e+08,10,6.542066e+07,4,6.113634e+07,511.91,184.23,52.5,148.62,65.42,61.14
1,Asia,30,2675,1.108310e+09,2382,3.470752e+08,147,1.033018e+08,132,2.675109e+08,15,1.054147e+08,13,2.850077e+08,1108.31,347.08,103.3,267.51,105.41,285.01
2,Europe,1,5,7.759970e+05,5,7.759970e+05,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00,0.78,0.78,0.0,0.00,0.00,0.00
3,Latin America and the Caribbean,21,936,3.383702e+08,835,1.107109e+08,59,4.109844e+07,51,1.052199e+08,0,0.000000e+00,6,8.134094e+07,338.37,110.71,41.1,105.22,0.00,81.34
4,Oceania,3,5,8.940537e+05,5,8.940537e+05,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00,0.89,0.89,0.0,0.00,0.00,0.00


## CSV outputs produced by this notebook

These files are saved under: `2_modelling/02_application/summary_statistics/`

1. `city_deprivation_80pct.csv`  
   – One row per city (passing ≥80% valid blocks), with:
   - Region, Country, City
   - TotalBlocks, ValidBlocks, PctValid
   - TotalPop, DeprivedPop, PctDeprived

2. `city_deprivation_with_sizeclass_80pct.csv`  
   – City-level table with city size classes:
   - Region, Country, City, CitySizeClass, TotalPop, DeprivedPop, PctDeprived

3. `country_deprivation_80pct.csv`  
   – Country-level aggregate:
   - Region, Country, TotalPop, DeprivedPop, PctDeprived

4. `regional_deprivation_80pct.csv`  
   – Region-level aggregate:
   - Region, TotalPop, DeprivedPop, PctDeprived, Countries, Cities

5. `region_sizeclass_deprivation_80pct.csv`  
   – Region × city-size-class aggregate:
   - Region, CitySizeClass, Cities, TotalPop, DeprivedPop, PctDeprived

6. `region_country_city.csv`  
   – Simple inventory (no QC filter):
   - Region, Country, City

7. `region_country_city_sizeclass_80pct.csv`  
   – Detailed city classification (QC-filtered):
   - Region, Country, City, Population, SizeClass

8. `region_summary_sizeclass_population_80pct.csv`  
   – Region-level summary with counts and population by city-size class:
   - Region, Countries, Cities, TotalPopulation (+ per-size-class city counts and populations, including *_Pop_Millions)
